# VCREASONER: Mechanistic Reasoning Traces Dataset Demo

This notebook demonstrates the **mechanistic reasoning traces dataset** from the VCREASONER paper.

## What is VCREASONER?

VCREASONER introduces a framework for **virtual cell reasoning** - multi agent system to understand and predict how biological perturbations (drugs, gene knockouts, etc.) affect cells at a mechanistic level.

## What does this dataset contain?

The dataset contains **structured mechanistic reasoning traces** that explain:
- How drugs/compounds affect specific cell types
- The molecular mechanisms involved (binding, enzymatic reactions, signaling cascades)
- Predicted phenotypic outcomes

Each trace includes:
1. **Human-readable reports** - Detailed mechanistic explanations in markdown
2. **Structured explanations** - Formalized action primitives describing molecular events
3. **Causal graphs (DAGs)** - Directed acyclic graphs showing cause-effect relationships

## Setup

In [1]:
import pandas as pd

## Load Dataset

In [2]:
df = pd.read_parquet('../data/vc_traces.parquet')

print(f"Dataset shape: {df.shape[0]:,} traces, {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")

Dataset shape: 18,950 traces, 6 columns

Columns: ['perturbation', 'question', 'report_text', 'explain', 'dag', 'explain_verified']


## Dataset Schema

| Column | Description |
|--------|-------------|
| `perturbation` | Dict containing the cellular context (cell type, mutations) and perturbation details (compound/gene, targets, mechanism) |
| `question` | The mechanistic question posed about the perturbation's effects |
| `report_text` | Human-readable mechanistic report in markdown format |
| `explain` | Structured explanation using action primitives (see below) |
| `dag` | Directed acyclic graph edges defining causal/correlative relationships |
| `explain_verified` | Verified/filtered version of the structured explanation |

In [3]:
print("Data types:")
print(df.dtypes)

Data types:
perturbation        object
question               str
report_text            str
explain                str
dag                    str
explain_verified       str
dtype: object


## Example Trace: Sodium Salicylate in Pancreatic Cancer Cells

Let's examine a complete example trace to understand the dataset structure.

In [4]:
# Select example trace (The index selected for Figure 2)
example_idx = 12563
example = df.iloc[example_idx]

print(f"Example index: {example_idx}")

Example index: 12563


### 1. Perturbation Context

The `perturbation` field contains two key components:
- **context**: The cellular environment (cell type, mutations, disease state)
- **perturbations**: The applied treatment (drug/compound with targets and mechanism)

In [5]:
perturbation = example['perturbation']

# Cell context
context = perturbation['context'][0]
print("=" * 60)
print("CELLULAR CONTEXT")
print("=" * 60)
print(f"Cell Type: {context['cell type']}")
print(f"Subtype: {context['subtype']}")
print(f"\nDescription:\n{context['description']}")

CELLULAR CONTEXT
Cell Type: HOP62
Subtype: disease cell

Description:
The source organ of cell line HOP62 is Lung. The key driver mutations of this cell line are listed as follow. Suppressor on gene B2M which causes None; Oncogene on gene KRAS which causes GoF; Suppressor on gene MSH6 which causes None; Suppressor on gene POLE which causes None; Oncogene on gene RAF1 which causes None; Oncogene on gene TERT which causes GoF; Suppressor on gene TP53 which causes LoF


In [6]:
# Perturbation details
pert = perturbation['perturbations'][0]
print("=" * 60)
print("PERTURBATION (Treatment)")
print("=" * 60)
print(f"Name: {pert['name']}")
print(f"Type: {pert['perturbation type']}")
print(f"Known Targets: {pert['known targets']}")
print(f"Mechanism: {pert['moa_type']}")
print(f"SMILES: {pert['smiles']}")

PERTURBATION (Treatment)
Name: venetoclax
Type: chemical
Known Targets: BCL2
Mechanism: unclear; inhibitor/antagonist
SMILES: CC1(CCC(=C(C1)C2=CC=C(C=C2)Cl)CN3CCN(CC3)C4=CC(=C(C=C4)C(=O)NS(=O)(=O)C5=CC(=C(C=C5)NCC6CCOCC6)[N+](=O)[O-])OC7=CN=C8C(=C7)C=CN8)C


### 2. Mechanistic Question

The question asks how the perturbation mechanistically and functionally influences the cell.

In [7]:
print("=" * 60)
print("QUESTION")
print("=" * 60)
# Show just the first part of the question (the core query)
question_lines = example['question'].split('\n')
print(question_lines[0])

QUESTION
**Q: How does the following perturbation influence the cell in the described context, mechanistically and functionally?**


### 3. Mechanistic Report

The `report_text` field contains a comprehensive, human-readable explanation of the mechanistic effects.

In [8]:
from IPython.display import Markdown, display

print("=" * 60)
print("MECHANISTIC REPORT (showing first sections)")
print("=" * 60)

# Display the report as formatted markdown
# Truncate to show structure without overwhelming output
report_sections = example['report_text'].split('## ')
truncated_report = '## '.join(report_sections[:4])  # Show first 3 sections

display(Markdown(truncated_report + "\n\n*[...truncated for brevity...]*"))

MECHANISTIC REPORT (showing first sections)


# Mechanistic Report: Venetoclax Treatment in HOP62 Lung Cancer Cells

## 1. Perturbation Description

**Venetoclax** is a selective, first-in-class BCL-2 inhibitor (BH3-mimetic) with high specificity for the BCL-2 protein. The compound functions as an ATP-competitive inhibitor with a binding affinity (Ki) of <0.01 nM for BCL-2, making it approximately 100-fold more selective for BCL-2 over related anti-apoptotic proteins BCL-XL and MCL-1. Venetoclax directly binds to the hydrophobic groove of BCL-2, mimicking the action of endogenous BH3-only proteins, and operates through **direct protein-protein interaction disruption** rather than enzymatic inhibition.

## 2. Cellular Context Analysis

**HOP62** is a lung adenocarcinoma cell line with a complex mutational landscape that creates a specific vulnerability profile:

### Key Driver Mutations:
- **KRAS GoF mutation**: Constitutively active RAS signaling driving proliferation and survival
- **TP53 LoF mutation**: Loss of p53-mediated apoptotic checkpoints
- **TERT GoF mutation**: Enhanced telomerase activity supporting unlimited replicative potential
- **RAF1 mutation**: Additional MAPK pathway dysregulation
- **Suppressor mutations**: B2M, MSH6, POLE losses affecting immune recognition and DNA repair

This mutational context creates **apoptosis-resistant** cancer cells that rely heavily on anti-apoptotic BCL-2 family proteins for survival, particularly in the context of oncogenic stress from hyperactive growth signaling.



*[...truncated for brevity...]*

### 4. Structured Explanation (Action Primitives)

The `explain` field contains a formalized representation using **action primitives** - standardized functions that describe mechanistic events.

In [9]:
print("=" * 60)
print("STRUCTURED EXPLANATION")
print("=" * 60)
print(example['explain'])

STRUCTURED EXPLANATION
set_context(cell_type="HOP62 lung adenocarcinoma", genotype="KRAS GoF, TP53 LoF, TERT GoF, B2M LoF, MSH6 LoF, POLE LoF, RAF1 mutation", disease="lung adenocarcinoma", prior_perturbation="none")
binds_to(id="n1", actor="venetoclax", target="BCL-2", affinity="<0.01 nM", via="BH3-binding groove occupancy")
modulates_molecule_activity(id="n2", target="BCL-2", direction="down", via="competitive BH3-mimetic inhibition")
modulates_molecule_activity(id="n3", target="BH3-only proteins", direction="up", via="release from BCL-2 sequestration")
modulates_molecule_activity(id="n4", target="BAX/BAK", direction="up", via="BH3-only protein activation")
modulates_complex(id="n5", members=["BAX", "BAK"], complex="BAX/BAK oligomers", direction="up", via="mitochondrial outer membrane pore formation")
localises_to(id="n6", entity="cytochrome c", from_loc="mitochondrial intermembrane space", to_loc="cytosol", mechanism="MOMP", via="n5")
modulates_complex(id="n7", members=["cytochrome 

### 5. Causal Graph (DAG)

The `dag` field defines the causal relationships between nodes in the explanation. Each edge specifies whether the relationship is **causal** (direct mechanism) or **correlative** (associated but not directly causing).

In [10]:
print("=" * 60)
print("DIRECTED ACYCLIC GRAPH (DAG)")
print("=" * 60)
print(example['dag'])

DIRECTED ACYCLIC GRAPH (DAG)
edge("n1", "n2", relation="causal")
edge("n2", "n3", relation="causal")
edge("n3", "n4", relation="causal")
edge("n4", "n5", relation="causal")
edge("n5", "n6", relation="causal")
edge("n6", "n7", relation="causal")
edge("n7", "n8", relation="causal")
edge("n8", "n9", relation="causal")
edge("n9", "n10", relation="causal")
edge("n9", "n11", relation="causal")
edge("n10", "n12", relation="causal")
edge("n11", "n12", relation="correlative")


### 6. Verified Explanation

The `explain_verified` field contains a refined version of the explanation after verification.

In [11]:
# Note that in the provided example, the verified explanation is the same as the original explanation
print("=" * 60)
print("VERIFIED EXPLANATION")
print("=" * 60)
print(example['explain_verified'])

VERIFIED EXPLANATION
set_context(cell_type="HOP62 lung adenocarcinoma", genotype="KRAS GoF, TP53 LoF, TERT GoF, B2M LoF, MSH6 LoF, POLE LoF, RAF1 mutation", disease="lung adenocarcinoma", prior_perturbation="none")
binds_to(id="n1", actor="venetoclax", target="BCL-2", affinity="<0.01 nM", via="BH3-binding groove occupancy")
modulates_molecule_activity(id="n2", target="BCL-2", direction="down", via="competitive BH3-mimetic inhibition")
modulates_molecule_activity(id="n3", target="BH3-only proteins", direction="up", via="release from BCL-2 sequestration")
modulates_molecule_activity(id="n4", target="BAX/BAK", direction="up", via="BH3-only protein activation")
modulates_complex(id="n5", members=["BAX", "BAK"], complex="BAX/BAK oligomers", direction="up", via="mitochondrial outer membrane pore formation")
localises_to(id="n6", entity="cytochrome c", from_loc="mitochondrial intermembrane space", to_loc="cytosol", mechanism="MOMP", via="n5")
modulates_complex(id="n7", members=["cytochrome c"

## Understanding Action Primitives

The structured explanations use a vocabulary of **action primitives** to describe molecular events. Here are the key primitives used in this dataset:

| Primitive | Description | Example |
|-----------|-------------|--------|
| `set_context()` | Establishes the biological context | Cell type, genotype, disease state |
| `binds_to()` | Drug-target or protein-protein binding | Drug binding to receptor with specified affinity |
| `modulates_molecule_activity()` | Changes in protein/enzyme activity | Inhibition or activation of an enzyme |
| `converts_substrate()` | Enzymatic reactions | Enzyme converting substrate to product |
| `regulates_expression()` | Gene expression changes | Transcription factor regulating target genes |
| `modulates_pathway_activity()` | Pathway-level effects | Activation/inhibition of signaling pathway |
| `induces_phenotype()` | Phenotypic outcomes | Growth inhibition, apoptosis, etc. |

## Visualizing the Mechanistic Chain

Let's parse the DAG and create a simple visualization of the causal chain.

In [15]:
import re

# Parse the explain field to extract node descriptions
explain_text = example['explain']
node_pattern = r'(\w+)\(id="(n\d+)"(.+?)\)'
context_pattern = r'set_context\((.+?)\)'

# Extract nodes
nodes = {}
for match in re.finditer(node_pattern, explain_text):
    action = match.group(1)
    node_id = match.group(2)
    params = match.group(3)
    
    # Extract key parameter for display
    if 'target=' in params:
        target = re.search(r'target="([^"]+)"', params)
        nodes[node_id] = f"{action}: {target.group(1) if target else ''}"[:40]
    elif 'actor=' in params:
        actor = re.search(r'actor="([^"]+)"', params)
        target = re.search(r'target="([^"]+)"', params)
        nodes[node_id] = f"{action}: {actor.group(1) if actor else ''} -> {target.group(1) if target else ''}"[:50]
    elif 'phenotype=' in params:
        phenotype = re.search(r'phenotype="([^"]+)"', params)
        nodes[node_id] = f"{action}: {phenotype.group(1) if phenotype else ''}"[:40]
    elif 'enzyme=' in params:
        enzyme = re.search(r'enzyme="([^"]+)"', params)
        product = re.search(r'product="([^"]+)"', params)
        nodes[node_id] = f"{action}: {enzyme.group(1) if enzyme else ''} -> {product.group(1) if product else ''}"[:50]
    elif 'regulator=' in params:
        regulator = re.search(r'regulator="([^"]+)"', params)
        nodes[node_id] = f"{action}: {regulator.group(1) if regulator else ''}"[:40]
    elif 'pathway=' in params:
        pathway = re.search(r'pathway="([^"]+)"', params)
        nodes[node_id] = f"{action}: {pathway.group(1) if pathway else ''}"[:40]
    else:
        nodes[node_id] = action[:50]

print("Nodes in the mechanistic explanation:")
print("-" * 60)
for node_id, desc in sorted(nodes.items(), key=lambda x: int(x[0][1:])):
    print(f"  {node_id}: {desc}")

Nodes in the mechanistic explanation:
------------------------------------------------------------
  n1: binds_to: BCL-2
  n2: modulates_molecule_activity: BCL-2
  n3: modulates_molecule_activity: BH3-only pr
  n4: modulates_molecule_activity: BAX/BAK
  n5: modulates_complex
  n6: localises_to
  n7: modulates_complex
  n8: modulates_molecule_activity: caspase-9
  n9: modulates_molecule_activity: caspase-3/7
  n10: post_translational_modification
  n11: regulates_expression: apoptotic signalin
  n12: induces_phenotype: apoptotic cell death


In [13]:
# Parse the DAG edges
dag_text = example['dag']
edge_pattern = r'edge\("(n\d+)",\s*"(n\d+)",\s*relation="(\w+)"\)'

print("\nCausal Graph Edges:")
print("-" * 60)
for match in re.finditer(edge_pattern, dag_text):
    source, target, relation = match.groups()
    arrow = "====>" if relation == "causal" else "- - ->"
    print(f"  {source} {arrow} {target}  [{relation}]")
    print(f"    ({nodes.get(source, source)} -> {nodes.get(target, target)})")
    print()


Causal Graph Edges:
------------------------------------------------------------
  n1 ====> n2  [causal]
    (binds_to: BCL-2 -> modulates_molecule_activity: BCL-2)

  n2 ====> n3  [causal]
    (modulates_molecule_activity: BCL-2 -> modulates_molecule_activity: BH3-only pr)

  n3 ====> n4  [causal]
    (modulates_molecule_activity: BH3-only pr -> modulates_molecule_activity: BAX/BAK)

  n4 ====> n5  [causal]
    (modulates_molecule_activity: BAX/BAK -> modulates_complex)

  n5 ====> n6  [causal]
    (modulates_complex -> localises_to)

  n6 ====> n7  [causal]
    (localises_to -> modulates_complex)

  n7 ====> n8  [causal]
    (modulates_complex -> modulates_molecule_activity: caspase-9)

  n8 ====> n9  [causal]
    (modulates_molecule_activity: caspase-9 -> modulates_molecule_activity: caspase-3/7)

  n9 ====> n10  [causal]
    (modulates_molecule_activity: caspase-3/7 -> post_translational_modificatio)

  n9 ====> n11  [causal]
    (modulates_molecule_activity: caspase-3/7 -> regula

## Summary

This dataset provides structured mechanistic reasoning traces for understanding drug effects on cells:

1. **Rich context**: Each trace includes cell type and perturbation as input
2. **Multiple representations**: Human-readable reports + machine-parseable action primitives
3. **Causal structure**: DAGs distinguish between causal and correlative relationships
4. **Verified traces**: Includes both initial and verified explanations

The action primitive vocabulary enables systematic analysis of mechanistic reasoning across thousands of perturbation-cell combinations.

In [14]:
# Quick dataset statistics
print("Dataset Statistics")
print("=" * 60)
print(f"Total traces: {len(df):,}")
print(f"\nAverage report length: {df['report_text'].str.len().mean():,.0f} characters")
print(f"Average explanation length: {df['explain'].str.len().mean():,.0f} characters")

Dataset Statistics
Total traces: 18,950

Average report length: 7,185 characters
Average explanation length: 1,720 characters
